# 06주 · 이산확률분포·시행 기준

공학통계학 CE202 · 2027학년도 1학기

슬라이드의 개념·숫자·풀이 순서와 연결한 **새 병행 실습**. 원본 Colab 복원본이 아닌 신규 자료

**목표**: 통계 원리와 도구의 기능 이해 → 좋은 AI 질문·지시 → 계산·가정·해석의 검토

**수업**: 2027-04-06 이산확률분포 (1) · 대면 / 2027-04-08 이산확률분포 (2) · 대면

[Excel 실습](https://raw.githubusercontent.com/steelpl/lecture/main/CE202-statistics-for-engineers/week06/CE202_W06.xlsx) · [주차 안내](https://github.com/steelpl/lecture/tree/main/CE202-statistics-for-engineers/week06)

슬라이드 확인 → 실행 전 예측 → 작은 손계산 → 코드 실행 → 조건 변경 → AI 요청 수정 → 결과 검토. 패키지 문법 암기보다 입력·정의·가정·출력의 의미 파악

셀 실행: Shift+Enter. 아래 코드의 None은 직접 채울 항목. 원본과 변경 사본의 결과를 구분

## 0. 실행 환경

필요한 패키지의 최소 버전 확인. 기본 코드와 보조 함수는 이번 주 실습에 필요한 도구. 세부 구현 암기 불필요

In [ ]:
# 최초 실행 시 필요한 버전만 설치. Colab에서 위에서 아래로 실행
import importlib.metadata as metadata, subprocess, sys
from packaging.version import Version
requirements = {"numpy": "1.26", "pandas": "2.2", "scipy": "1.16", "matplotlib": "3.10", "openpyxl": "3.1"}
for package, minimum in requirements.items():
    try:
        outdated = Version(metadata.version(package)) < Version(minimum)
    except metadata.PackageNotFoundError:
        outdated = True
    if outdated:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", f"{package}>={minimum}"])
print("실행 환경 준비 완료. 이미 불러온 패키지를 업데이트한 경우 런타임 재시작 후 처음부터 실행")

In [ ]:
import math, tempfile, base64, zipfile, io
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
WORK = Path(tempfile.mkdtemp(prefix="ce202-"))
rng = np.random.default_rng(2027)

def vector(values, minimum=1):
    x = np.asarray(values, dtype=float)
    if x.ndim != 1 or len(x) < minimum or not np.isfinite(x).all():
        raise ValueError("유한한 1차원 수치 자료와 최소 표본수 확인 필요")
    return x

def probabilities(values):
    p = vector(values)
    if np.any(p < 0) or not np.isclose(p.sum(), 1):
        raise ValueError("확률의 비음수성·합계 1 확인 필요")
    return p

def mean_ci(values, confidence=0.95):
    x = vector(values, 2)
    if not 0 < confidence < 1:
        raise ValueError("신뢰수준은 0과 1 사이")
    se = x.std(ddof=1) / np.sqrt(len(x))
    q = stats.t.ppf((1 + confidence) / 2, df=len(x)-1)
    return {"n": len(x), "mean": x.mean(), "se": se,
            "df": len(x)-1, "low": x.mean()-q*se, "high": x.mean()+q*se}

def wilson_interval(successes, n, confidence=0.95):
    if not isinstance(n, (int, np.integer)) or n <= 0 or not 0 <= successes <= n:
        raise ValueError("0 ≤ 성공수 ≤ 양의 정수 표본수 확인 필요")
    if int(successes) != successes or not 0 < confidence < 1:
        raise ValueError("정수 성공수·신뢰수준 확인 필요")
    z = stats.norm.ppf((1+confidence)/2)
    phat = successes / n
    den = 1 + z*z/n
    center = (phat + z*z/(2*n)) / den
    half = z / den * np.sqrt(phat*(1-phat)/n + z*z/(4*n*n))
    return center-half, center+half

def anova_summary(groups):
    arrays = [vector(g, 2) for g in groups]
    if len(arrays) < 2:
        raise ValueError("두 집단 이상 필요")
    n = np.array([len(g) for g in arrays]); means = np.array([g.mean() for g in arrays])
    grand = np.concatenate(arrays).mean()
    ss_between = np.sum(n*(means-grand)**2)
    ss_within = sum(np.sum((g-g.mean())**2) for g in arrays)
    df_between = len(arrays)-1; df_within = n.sum()-len(arrays)
    if ss_within <= 0:
        raise ValueError("집단 내 변동이 0이면 일반적인 F 계산 불가")
    F = (ss_between/df_between) / (ss_within/df_within)
    return {"ss_between": ss_between, "ss_within": ss_within,
            "df_between": df_between, "df_within": df_within,
            "F": F, "p": stats.f.sf(F, df_between, df_within)}

def regression(x, y):
    x = vector(x, 3); y = vector(y, 3)
    if len(x) != len(y) or np.ptp(x) == 0:
        raise ValueError("같은 길이·서로 다른 x 값 확인 필요")
    fit = stats.linregress(x, y)
    fitted = fit.intercept + fit.slope*x
    residual = y-fitted
    sse = np.sum(residual**2); sst = np.sum((y-y.mean())**2)
    if sst <= 0:
        raise ValueError("y 변동이 0이면 일반적인 R² 해석 불가")
    return {"slope": fit.slope, "intercept": fit.intercept, "r2": 1-sse/sst,
            "sse": sse, "mse_residual": sse/(len(x)-2), "mse_prediction": sse/len(x),
            "p": fit.pvalue, "stderr": fit.stderr, "fitted": fitted, "residual": residual}


## S01. 균등·이항·초기하

**슬라이드 연결**: 2026_S1_Statistics_Wk6a_Dist1.pptx p11, 18, 23, 26, 30, 32

**자료 구분**: 원본 예제·수치 기반

**먼저 예측**: 출생월이 같은 확률이라는 가정과 실제 출생분포는 같은가?

계산 전에 분모·단위·조건을 한 줄로 기록

In [ ]:
print("월 균등 가정의 평균:",np.mean(np.arange(1,13)))
print("불량률 .05, 10개 중 정확히 2개:",stats.binom.pmf(2,n=10,p=0.05))
print("32개 중 16개 성공, 비복원 16개 중 8개:",stats.hypergeom.pmf(8,M=32,n=16,N=16))
assert np.isclose(stats.binom.sf(1,10,0.05),1-stats.binom.cdf(1,10,0.05))

**조건 변경**: 비복원·복원 여부만 바꾸고 초기하·이항 분산 비교

위 코드를 복사한 셀에서 한 조건씩 변경. 원본 셀은 보존

**AI 요청에 넣을 정보**: 성공 정의·전체 시행수·비복원 여부·꼬리의 포함 조건 명시

**결과 검토**: P(X≥2)는 1−P(X≤1). scipy hypergeom 모수 이름 확인

실행 결과·손계산 근거·수정 지시·채택 또는 거절 이유 기록

In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


## S02. 푸아송·기하·음이항의 기준

**슬라이드 연결**: 2026_S1_Statistics_Wk6b_Dist2.pptx p11, 19, 37, 40

**자료 구분**: 원본 예제·수치 기반

**먼저 예측**: 정확히 5장 회수와 5번째 회수가 10번째 시행인 사건은 같은가?

계산 전에 분모·단위·조건을 한 줄로 기록

In [ ]:
print("시간당 평균1 전화, 정확히2:",stats.poisson.pmf(2,mu=1))
p=0.3
fixed_10=stats.binom.pmf(5,n=10,p=p)
fifth_on_10=stats.nbinom.pmf(5,n=5,p=p) # 다섯 번째 성공 전 실패 5회
print("10장 중 정확히5 회수:",fixed_10,"5번째 회수가 10번째:",fifth_on_10)
print("500장 회수까지 기대 발송수:",500/p)
print("A팀(.85)이 5차전에서 4승 확정:",math.comb(4,3)*0.85**4*(1-0.85))
assert np.isclose(fifth_on_10,0.5*fixed_10)
print("geom의 첫 성공 시행수=1:",stats.geom.pmf(1,p=p))

**조건 변경**: SciPy 음이항의 값이 전체 시행수인지 실패수인지 바꿔 검토

위 코드를 복사한 셀에서 한 조건씩 변경. 원본 셀은 보존

**AI 요청에 넣을 정보**: 전체 시행수/실패수·정지 규칙·성공확률 일정·독립 가정을 명시

**결과 검토**: 단위시간이 바뀌면 푸아송 평균도 같은 비율로 변경

실행 결과·손계산 근거·수정 지시·채택 또는 거절 이유 기록

In [ ]:
# 변경한 코드·확인할 식을 이 셀에 추가


## 주차 확인

분포·시행 기준·원본 전화 확률 기록

입력 항목:

- `poisson_lambda`: 수치·분모·자유도 등 해당 계산값
- `poisson_exactly_two`: 수치·분모·자유도 등 해당 계산값
- `binomial_and_stopping_events_same`: 참/거짓 판단 · Python `True`/`False`

공개 확인용 항목. 결과값만 복사한 경우 학습 완료로 판단 불가. 분석 가정·자료 출처·추론 범위는 아래 기록과 함께 검토

In [ ]:
answers = {'poisson_lambda': None, 'poisson_exactly_two': None, 'binomial_and_stopping_events_same': None}


In [ ]:
def check_answers(values):
    expected = {'poisson_lambda': 1.0, 'poisson_exactly_two': 0.18393972058572114, 'binomial_and_stopping_events_same': False}
    pending, wrong = [], []
    for key, target in expected.items():
        got = values.get(key)
        if got is None:
            pending.append(key)
        elif isinstance(target, bool):
            if not isinstance(got, (bool, np.bool_)) or bool(got) != target:
                wrong.append(key)
        elif isinstance(target, str):
            if not isinstance(got, str) or got.strip().lower() != target:
                wrong.append(key)
        elif isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, float, np.number)) or not np.isfinite(got) or not np.isclose(got, target, rtol=1e-6, atol=1e-8):
            wrong.append(key)
    if pending:
        print("미완료:", ", ".join(pending))
    if wrong:
        print("재검토:", ", ".join(wrong))
    if not pending and not wrong:
        print("공개 계산·개념 확인 통과. 해석·가정·AI 검토 기록은 별도 확인 필요")
    return {"pending": pending, "wrong": wrong, "passed": not pending and not wrong}

check_result = check_answers(answers)

## AI 요청·검토 기록

다음 항목에 직접 기록. 글자 수 자동 채점 없음

1. 분석 질문·관측단위·자료 출처·변수·단위
2. 최초 AI 요청: 목적·조건·방법·출력 형식·금지할 가정
3. 실행 전 예상·손계산 근거
4. AI 출력 중 의심한 부분·재현 결과·수정 요청
5. 최종 결과와 원본의 일치 여부·분석 가정·불확실성·한계
6. AI 없이 설명 가능한 핵심 원리 한 가지

채택·수정·거절 중 하나를 선택하고 근거 제시. p값·R² 하나만으로 결론 확정 금지

In [ ]:
review_log = {
    "분석 질문·자료·단위": None,
    "최초 AI 요청": None,
    "예상·손계산": None,
    "의심한 부분·수정 요청": None,
    "최종 판단·가정·한계": None,
    "내 설명": None,
}
